In [1]:
from dotenv import load_dotenv
import yfinance as yf
import pandas as pd
import time
import pyodbc
import os

load_dotenv()

In [2]:
def get_or_create_security(cursor, ticker, name, sector=None):
    cursor.execute("SELECT security_id FROM securities WHERE ticker = ?", ticker)
    row = cursor.fetchone()
    if row:
        return row.security_id

    cursor.execute(
        "INSERT INTO securities (ticker, name, sector) OUTPUT INSERTED.security_id VALUES (?, ?, ?)",
        ticker, name, sector
    )
    return cursor.fetchone().security_id

True

In [4]:
sa_password = os.environ['MSSQL_SA_PASSWORD']
tickers = ['MSFT', 'AAPL', 'GOOG', 'NVDA', 'AMD', 
               'SYY', 'F', 'PLUG', 'SOFI', 'TSLA', 'SMCI', 
               'AMZN', 'RIG', 'CHWY', 'CRWV', 'AMC', 'UBER',
               'HPE', 'VZ']

In [5]:
conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost,1433;"
    "DATABASE=MarketData;"
    "UID=sa;"
    f"PWD={sa_password};"
    "TrustServerCertificate=yes;"
)
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

In [13]:
for ticker in tickers:
    df = yf.Ticker(ticker).history(period="1y", interval="1d")
    df = df.reset_index() # turns the Date index into a column
    df = df.sort_values("Date") # ensure chronological order for rolling calcs

    if df.empty:
        print(f"WARNING: no data returned for {ticker}")
        continue

    df["daily_return"] = df["Close"].pct_change()
    df["moving_avg_50d"] = df["Close"].rolling(window=50).mean()
    df["rolling_volatility"] = df["daily_return"].rolling(window=50).std()

    df = df.astype(object).where(pd.notnull(df), None)

    info = yf.Ticker(ticker).info
    name = info.get("longName", ticker)
    sector = info.get("sector")
    security_id = get_or_create_security(cursor, ticker, name, sector)

    for _, row in df.iterrows():
        cursor.execute(
                        "INSERT INTO prices (security_id, trade_date, open_price, high_price, low_price, close_price, volume, daily_return, moving_avg_50d, rolling_volatility) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                        security_id, row['Date'], row['Open'], row['High'], row['Low'], row['Close'], row['Volume'], row['daily_return'], row["moving_avg_50d"], row["rolling_volatility"]
        )


In [14]:
conn.commit()